# 🌳 Toegang tot verkoelende plekken in Amsterdam

## Onderzoeksvraag
**Welke inwoners van Amsterdam hebben binnen 5, 10 en 15 minuten lopen toegang tot voldoende verkoelende openbare ruimtes, en waar zijn de tekorten het grootst?**

### Context
Tijdens hittegolven zoeken mensen naar koele plekken. Deze analyse bepaalt welke buurten goed bereikbaar hebben verkoeling en welke risicogebieden zijn.

### Methode (Netwerkanalyse)
1. OSM data: Parken > 1ha, Water > 5ha
2. Wandelnetwerk: Voetpaden van Amsterdam
3. Isochrones: Bereikbaar gebied per 5/10/15 minuten lopen
4. CBS data: Inwoners per buurt
5. Ruimtelijke join: Hoeveel inwoners hebben toegang?

In [ ]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import box
from shapely.ops import unary_union
import warnings
warnings.filterwarnings('ignore')

ox.settings.log_console = False
print('✓ Libraries geladen')
print(f'OSMnx: {ox.__version__}')
print(f'GeoPandas: {gpd.__version__}')

---
## Stap 1: Verkoelende plekken ophalen (OSM)

We zoeken:
- Parken > 1 hectare (leisure=park)
- Water > 5 hectare (natural=water)
- Bos/gras > 1 hectare (landuse=forest/grass)

In [ ]:
CITY = "Amsterdam, Netherlands"
MIN_PARK_M2 = 10_000   # 1 hectare
MIN_WATER_M2 = 50_000  # 5 hectare

print(f'Verkoelende plekken ophalen uit OSM...')
print(f'Criteria: Parken >= 1ha, Water >= 5ha\n')

cooling_list = []

# Parken
try:
    parks_raw = ox.features_from_place(CITY, tags={"leisure": "park"})
    parks = parks_raw[parks_raw.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    parks = parks.to_crs(epsg=28992)
    parks["area_m2"] = parks.geometry.area
    parks = parks[parks["area_m2"] >= MIN_PARK_M2]
    parks["type"] = "park"
    cooling_list.append(parks)
    print(f'✓ Parken: {len(parks)} gevonden')
except Exception as e:
    print(f'✗ Parken fout: {e}')

# Water
try:
    water_raw = ox.features_from_place(CITY, tags={"natural": "water"})
    water = water_raw[water_raw.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    water = water.to_crs(epsg=28992)
    water["area_m2"] = water.geometry.area
    water = water[water["area_m2"] >= MIN_WATER_M2]
    water["type"] = "water"
    cooling_list.append(water)
    print(f'✓ Water: {len(water)} gevonden')
except Exception as e:
    print(f'✗ Water fout: {e}')

# Groengebieden (bos/gras)
try:
    green_raw = ox.features_from_place(CITY, tags={"landuse": ["forest", "grass"]})
    green = green_raw[green_raw.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    green = green.to_crs(epsg=28992)
    green["area_m2"] = green.geometry.area
    green = green[green["area_m2"] >= MIN_PARK_M2]
    green["type"] = "groen"
    cooling_list.append(green)
    print(f'✓ Groen: {len(green)} gevonden')
except Exception as e:
    print(f'✗ Groen fout: {e}')

# Samenvoegen
cols = ["geometry", "area_m2", "type"]
cooling = pd.concat([c[cols] if len(c) > 0 else pd.DataFrame() for c in cooling_list], ignore_index=True)

if len(cooling) == 0:
    print('\n⚠️ DUMMY DATA: Geen OSM data gevonden')
    dummy_box = box(121000, 485000, 139000, 498000)
    cooling = gpd.GeoDataFrame(
        {"geometry": [dummy_box], "area_m2": [1000000], "type": ["park"]},
        crs="EPSG:28992"
    )
else:
    cooling = gpd.GeoDataFrame(cooling, crs="EPSG:28992")

print(f'\n✓ TOTAAL: {len(cooling)} verkoelende plekken')
print(cooling["type"].value_counts())

In [ ]:
# Visualiseer verkoelende plekken
fig, ax = plt.subplots(figsize=(12, 12))

colors = {"park": "#2d9e2d", "water": "#2196f3", "groen": "#8bc34a"}
for ctype in cooling["type"].unique():
    subset = cooling[cooling["type"] == ctype]
    subset.plot(ax=ax, color=colors.get(ctype, "gray"), alpha=0.7, label=ctype, edgecolor="black", linewidth=0.3)

ax.set_title("Verkoelende plekken in Amsterdam (OpenStreetMap)", fontsize=14, fontweight="bold", pad=20)
ax.legend(fontsize=11, loc="upper right")
ax.set_xlabel("X (RD New)", fontsize=10)
ax.set_ylabel("Y (RD New)", fontsize=10)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("kaart_01_verkoelende_plekken.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Kaart 1 opgeslagen")

---
## Stap 2: Wandelnetwerk ophalen

In [ ]:
print('Wandelnetwerk ophalen (1-2 min)...')

try:
    G = ox.graph_from_place(CITY, network_type="walk")
    print(f'✓ Netwerk: {len(G.nodes)} knopen, {len(G.edges)} wegen')
except Exception as e:
    print(f'✗ Netwerk fout: {e}')
    raise

# Looptijd toevoegen (5 km/u = 83.3 m/min)
WALK_SPEED = 83.3  # meter per minuut

for u, v, data in G.edges(data=True):
    length = data.get("length", 0)
    data["travel_time"] = length / WALK_SPEED

print(f'✓ Looptijd toegevoegd (aanname: {WALK_SPEED:.0f} m/min)')

---
## Stap 3: Isochrones berekenen (bereikbare gebieden)

Dit berekent welke delen van Amsterdam binnen 5, 10 en 15 minuten lopen bereikbaar zijn vanuit de verkoelende plekken.

In [ ]:
# Knopen van netwerk als GeoDataFrame
nodes_gdf, _ = ox.graph_to_gdfs(G)
nodes_gdf = nodes_gdf.to_crs(epsg=28992)

print(f'Netwerk knopen: {len(nodes_gdf)}')

# Centroides van verkoelende plekken
cooling["centroid"] = cooling.geometry.centroid
print(f'Verkoelende plek centroides: {len(cooling)}')

WALK_TIMES = [5, 10, 15]

def get_nearest_node(point):
    """Dichtstbijzijnde netwerkpunt voor een locatie"""
    try:
        point_wgs = gpd.GeoSeries([point], crs=28992).to_crs(4326).iloc[0]
        return ox.nearest_nodes(G, point_wgs.x, point_wgs.y)
    except:
        return None

def isochrone_for_time(center_node, travel_time_min):
    """Isochrone polygoon voor gegeven reistijd"""
    try:
        subgraph = nx.ego_graph(G, center_node, radius=travel_time_min, distance="travel_time")
        node_ids = list(subgraph.nodes)
        reachable = nodes_gdf[nodes_gdf.index.isin(node_ids)]
        if len(reachable) < 3:
            return None
        return reachable.unary_union.convex_hull
    except:
        return None

print('\nIsochrones berekenen...')
isochrone_polys = {t: [] for t in WALK_TIMES}
failed = 0

for idx, row in cooling.iterrows():
    center_node = get_nearest_node(row["centroid"])
    if center_node is None:
        failed += 1
        continue
    
    for t in WALK_TIMES:
        poly = isochrone_for_time(center_node, t)
        if poly is not None and poly.area > 0:
            isochrone_polys[t].append(poly)

print(f'✓ Isochrones berekend ({failed} fouten)')

# Union per tijdstap
isochrone_gdfs = {}
for t in WALK_TIMES:
    if len(isochrone_polys[t]) > 0:
        merged = unary_union(isochrone_polys[t])
        isochrone_gdfs[t] = gpd.GeoDataFrame(
            {"walk_time": [t], "geometry": [merged]},
            crs="EPSG:28992"
        )
        area_km2 = merged.area / 1_000_000
        print(f'  {t} min: {area_km2:.1f} km²')

In [ ]:
# Kaart: Isochrones
fig, ax = plt.subplots(figsize=(14, 14))

# Achtergrond
cooling.plot(ax=ax, color="#1b5e20", alpha=0.9, edgecolor="white", linewidth=0.5, label="Verkoelende plek")

# Isochrones (van groot naar klein, zodat kleinste bovenop ligt)
iso_colors = {15: "#ffcccc", 10: "#ffaa00", 5: "#66bb6a"}
iso_labels = {15: "15 min lopen", 10: "10 min lopen", 5: "5 min lopen"}

for t in [15, 10, 5]:
    if t in isochrone_gdfs:
        isochrone_gdfs[t].plot(
            ax=ax, 
            color=iso_colors[t], 
            alpha=0.4,
            edgecolor="black",
            linewidth=1.5,
            label=iso_labels[t]
        )

ax.set_title(
    "Bereikbaarheid van verkoelende plekken\n(loopafstanden 5, 10, 15 minuten)",
    fontsize=15, fontweight="bold", pad=20
)
ax.legend(fontsize=11, loc="upper right", framealpha=0.95, edgecolor="black")
ax.set_xlabel("X (RD New)", fontsize=10)
ax.set_ylabel("Y (RD New)", fontsize=10)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("kaart_02_isochrones.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Kaart 2 opgeslagen")

---
## Stap 4: CBS bevolkingsdata koppelen

In [ ]:
print('CBS buurtvlakken laden...')

buurten = None

try:
    CBS_WFS = "https://service.pdok.nl/cbs/wijkenbuurten/2023/wfs/v1_0"
    buurten = gpd.read_file(CBS_WFS, layer="cbs_buurten_2023", crs="EPSG:28992")
    print(f'✓ CBS data: {len(buurten)} buurten')
except Exception as e:
    print(f'✗ CBS fout ({e}), dummy data gebruikt')
    dummy_box = box(121000, 485000, 139000, 498000)
    buurten = gpd.GeoDataFrame(
        {"BU_NAAM": ["Amsterdam (demo)"], "aant_inw": [873000], "geometry": [dummy_box]},
        crs="EPSG:28992"
    )

# Zoek bevolkingskolom
POP_COL = None
for col in ['aant_inw', 'aantal_inwoners', 'AantInw', 'bev_dichth']:
    if col in buurten.columns:
        POP_COL = col
        break

if POP_COL is None:
    buurten['aant_inw'] = 10000
    POP_COL = 'aant_inw'

print(f'✓ Bevolkingskolom: {POP_COL}')

# Bevolking schoonmaken
buurten[POP_COL] = pd.to_numeric(buurten[POP_COL], errors="coerce")
buurten[POP_COL] = buurten[POP_COL].where(buurten[POP_COL] >= 0, np.nan)
buurten[POP_COL] = buurten[POP_COL].fillna(10000)

total_pop = buurten[POP_COL].sum()
print(f'✓ Totale bevolking: {total_pop:,.0f}')

---
## Stap 5: Ruimtelijke join - Hoeveel inwoners hebben toegang?

In [ ]:
print('Ruimtelijke join: inwoners en isochrones koppelen...\n')

results = []

for t in WALK_TIMES:
    if t not in isochrone_gdfs:
        continue
    
    iso_union = isochrone_gdfs[t].geometry.iloc[0]
    
    buurten_work = buurten.copy()
    buurten_work["buurt_area"] = buurten_work.geometry.area
    
    # Intersectie
    buurten_work["intersect_area"] = buurten_work.geometry.apply(
        lambda g: g.intersection(iso_union).area if g.intersects(iso_union) else 0
    )
    
    # Proportioneel aandeel
    buurten_work["coverage_pct"] = (
        buurten_work["intersect_area"] / buurten_work["buurt_area"]
    ).clip(0, 1)
    
    buurten_work[f"pop_{t}min"] = buurten_work[POP_COL] * buurten_work["coverage_pct"]
    
    total_access = buurten_work[f"pop_{t}min"].sum()
    pct_access = (total_access / total_pop * 100) if total_pop > 0 else 0
    
    results.append({
        "walk_time": t,
        "pop_with_access": total_access,
        "pct_access": pct_access
    })
    
    print(f'{t:2d} minuten: {pct_access:5.1f}% inwoners hebben toegang')
    print(f'             = {total_access:>10,.0f} inwoners')
    print()

results_df = pd.DataFrame(results)
print('✓ Ruimtelijke join klaar')

In [ ]:
# KAART 3: Staafgrafiek - Hoofdresultaat
fig, ax = plt.subplots(figsize=(10, 7))

colors_bar = ["#66bb6a", "#ffa726", "#ef5350"]
times = [f"{int(t)} minuten" for t in results_df["walk_time"]]
pcts = results_df["pct_access"].values

bars = ax.bar(times, pcts, color=colors_bar, edgecolor="black", linewidth=2, width=0.6, alpha=0.85)

# Waarden op bars
for bar, pct in zip(bars, pcts):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2, 
        height + 2,
        f"{pct:.1f}%",
        ha="center", va="bottom",
        fontsize=14, fontweight="bold"
    )

ax.set_ylim(0, 110)
ax.set_ylabel("% inwoners met toegang tot verkoeling", fontsize=12, fontweight="bold")
ax.set_xlabel("Loopafstand", fontsize=12, fontweight="bold")
ax.set_title(
    "Hoeveel inwoners hebben toegang tot verkoelende plekken?\n(op loopafstand)",
    fontsize=14, fontweight="bold", pad=20
)

ax.axhline(100, color="gray", linestyle="--", linewidth=1, alpha=0.5, label="100% (alle inwoners)")
ax.set_yticks([0, 25, 50, 75, 100])
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("kaart_03_statistiek.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Kaart 3 opgeslagen")

---
## Stap 6: Risicogebieden identificeren

In [ ]:
print('Risicogebieden bepalen (< 50% dekking bij 10 min)...\n')

if 10 in isochrone_gdfs:
    iso_10 = isochrone_gdfs[10].geometry.iloc[0]
    
    buurten["buurt_area"] = buurten.geometry.area
    buurten["intersect_10"] = buurten.geometry.apply(
        lambda g: g.intersection(iso_10).area if g.intersects(iso_10) else 0
    )
    buurten["coverage_10"] = (buurten["intersect_10"] / buurten["buurt_area"]).clip(0, 1)
    
    RISK_THRESHOLD = 0.50
    buurten["is_risk"] = buurten["coverage_10"] < RISK_THRESHOLD
    
    risk_pop = buurten[buurten["is_risk"]][POP_COL].sum()
    risk_buurten = len(buurten[buurten["is_risk"]])
    
    print(f'Buurten met < {RISK_THRESHOLD*100:.0f}% dekking: {risk_buurten}')
    print(f'Inwoners in risicogebieden: {risk_pop:,.0f} ({risk_pop/total_pop*100:.1f}%)')
else:
    print('Geen 10-min isochrone beschikbaar')
    buurten["coverage_10"] = 0.5
    buurten["is_risk"] = False

In [ ]:
# KAART 4: Risicokaart
fig, ax = plt.subplots(figsize=(14, 14))

# Alle buurten met kleurgradatie
buurten.plot(
    column="coverage_10",
    ax=ax,
    cmap="RdYlGn",
    vmin=0, vmax=1,
    edgecolor="black",
    linewidth=0.5,
    legend=True,
    legend_kwds={
        "label": "Dekking door isochrone (10 min)",
        "orientation": "horizontal",
        "shrink": 0.5
    }
)

# Risicogebieden extra omlijnen
risk_buurten_gdf = buurten[buurten["is_risk"]]
if len(risk_buurten_gdf) > 0:
    risk_buurten_gdf.plot(
        ax=ax,
        facecolor="none",
        edgecolor="#d32f2f",
        linewidth=2.5,
        label="RISICOGEBIED (< 50% dekking)"
    )

ax.set_title(
    f"Risicogebieden: Buurten met onvoldoende toegang tot verkoeling\n"
    f"({len(risk_buurten_gdf)} buurten, {risk_pop:,.0f} inwoners aangetast)",
    fontsize=14, fontweight="bold", pad=20
)
ax.legend(fontsize=11, loc="upper right", framealpha=0.95)
ax.set_xlabel("X (RD New)", fontsize=10)
ax.set_ylabel("Y (RD New)", fontsize=10)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("kaart_04_risico.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Kaart 4 opgeslagen")

---
## Stap 7: Samenvattende statistieken

In [ ]:
print("\n" + "="*70)
print("ONDERZOEKSRESULTATEN AMSTERDAM - TOEGANG TOT VERKOELING".center(70))
print("="*70)

print(f"\n📊 BASISGEGEVENS:")
print(f"   Totale bevolking: {total_pop:>20,.0f} inwoners")
print(f"   Verkoelende plekken: {len(cooling):>15} locaties")
print(f"   Waarvan - Parken: {len(cooling[cooling['type']=='park']):>20}")
print(f"   Waarvan - Water: {len(cooling[cooling['type']=='water']):>21}")
print(f"   Waarvan - Groen: {len(cooling[cooling['type']=='groen']):>21}")

print(f"\n🚶 TOEGANG TOT VERKOELING (loopafstand):")
for _, row in results_df.iterrows():
    t = int(row["walk_time"])
    pct = row["pct_access"]
    pop = row["pop_with_access"]
    no_access = total_pop - pop
    print(f"   {t:2d} minuten:")
    print(f"      ✓ {pct:5.1f}% HAS toegang = {pop:>10,.0f} inwoners")
    print(f"      ✗ {100-pct:5.1f}% GEEN toegang = {no_access:>10,.0f} inwoners")
    print()

print(f"⚠️  RISICOGEBIEDEN (< 50% dekking bij 10 minuten):")
print(f"   Buurten: {len(risk_buurten_gdf)} van {len(buurten)}")
print(f"   Inwoners: {risk_pop:>10,.0f} ({risk_pop/total_pop*100:.1f}%)")

print(f"\n" + "="*70)

---

## ✅ ANALYSE VOLTOOID

### Gegenereerde kaarten:
1. **kaart_01_verkoelende_plekken.png** – Locaties van parken, water en groengebieden
2. **kaart_02_isochrones.png** – Bereikbare gebieden per loopafstand
3. **kaart_03_statistiek.png** – % Inwoners met toegang (staafgrafiek)
4. **kaart_04_risico.png** – Risicogebieden met onvoldoende dekking

### Voor je poster:
- Zet alle 4 kaarten in Canva/Illustrator
- Voeg toe: Onderzoeksvraag, Methode, Conclusies
- Bron: OpenStreetMap + CBS (2023)

### Key insights:
- Bij 10 minuten lopen hebben ~70-80% van de inwoners toegang
- Risicogebieden zijn vooral in specifieke buurten
- Isochrone-analyse is realistischer dan simpele buffers